# PerovStats demo notebook
### This notebook is for demonstrating the processes involved in PerovStats, from deciding on configuration options and running the modules.

If you encounter bugs please make a report of them as a [GitHub issue](https://github.com/AFM-SPM/PerovStats/issues/new?template=bug_report.yml). Please provide as much info as you can, if you don't know some specific details that is ok.

Alternatively, email t.allwood@sheffield.ac.uk - I will get back to you as soon as I can

#### Before running this notebook please read the installation instructions (in the `/docs/` folder) to make sure the project dependencies are set up for the program.

#### To ensure the sliders work correctly please run this notebook one cell at a time rather than clicking the 'Run All' button

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import copy

from loguru import logger
from pathlib import Path
import yaml
from topostats.io import LoadScans
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from ipywidgets import interact, FloatSlider, Layout
from IPython.display import display

sys.path.append(os.path.abspath(os.path.join('..')))
from src.perovstats.core.classes import PerovStats, ImageData
from src.perovstats.core.io import save_to_csv, save_config, save_images
from src.perovstats.core.image_processing import normalise_array
from src.perovstats.cli import setup_logger, deep_merge
from src.perovstats.fourier import split_frequencies
from src.perovstats.segmentation import segment_image
from src.perovstats.smears import find_smear_areas
from src.perovstats.processing import format_time, completion_message
from src.perovstats.grains import find_grains

---
## Setup

#### These options can be changed to suit your needs. the `output_dir` folder will contain a selection of images from various stages in the process as well as the `.csv` files generated.
In the full version of the notebook, `img_file` will be replaced with `img_files` and will need to point to a directory containing .spm files rather than a single specific file.
There are two options for files to test this program on. The first example contains smear areas and the second does not.

You are of course able to test with your own AFM files by replacing the path in the double quotes in `img_file` with your own filepath.

In [ ]:
img_file = [Path("../example images/4_c60_perovonsil_ref_10um.PFQNM.spm")]
# img_file = [Path("../example images/TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm")]
output_dir = Path("./output")
config_path = Path("../src/perovstats/default_config.yaml")


### Custom variables

The default settings will work on a number of images, but sometimes you may want to edit the following variables for better results.

##### `file_ext`

- Identify the file extension of the scan being used in double quotes, e.g. `".spm"`

##### `run_splitting`

- If you are processing images that don't have a background material you can disable frequency splitting by changing this variable to `False`. This will bypass the frequency splitting process and move on to grain finding.

##### `cutoff_bounds`

- The `cutoff_bounds` variable is used for finding an ideal frequency to split an image by when splitting perovskite grains from the surface they're on. If a cutoff cannot be found with the current bounds increasing them up or down can help this issue. The difference between each bound does not affect the speed of this process, but a smaller difference will allow for higher accuracy when splitting an image.

- When a cutoff is found the value is logged in the output, you can use this to further fine-tune the cutoff bounds if higher accuracy is required.

##### `min_rms`

- This is the variable currently used to adjust the frequency chosen when performing a fourier transform (separation of silicon and perovskite bumps)
- This checks the roughness of the high-pass (perovskite) image and accepts the current split if the image has a roughness value similar to what we would expect a perovskite material to have.
- The default value is 10 which has been working well for images used in testing
- If silicon 'mountains' are still visible in the high-passed image lower the `min_rms` value
- If the perovskite grains don't look defined enough increase the `min_rms` value
- This does not have to be a full number and it is recommended to only adjust the value by a maximum of 2 inbetween runs, I don't expect you'll ever want a value under ~7 or over ~15


#### There are two ways of segmenting perovskite grains depending on whether you want to prioritise speed or accuracy
- `traditional`: As the name suggests, this option uses traditional, hard-coded, segmentation methods to outline grain borders. This method is fairly quick (only taking a few seconds per image) but can come with inaccuracies throughout the mask and is prone to big issues if config parameters are not set correctly.
- `cellpose`: Cellpose is a machine learning model used for grain segmentation in images similar to a perovskite scan. It has a much higher accuracy than traditional methods and does not require any configuration options to be entered. This takes much longer (a few minutes) but running PerovStats on a computer with an Nvidia GPU will greatly decrease this time.

For the demonstration purposes of this notebook both methods will be run to show the differences in speed and accuracy between the two. In the full run in the other notebook you will have a space to select the method you'd like for a run.

##### If traditional segmentation has been selected a sensitivity parameter will need to be configured for the segmentation.
    
- **If cellpose segmentation has been selected you can ignore this variable.**

`threshold_offset`
- This is the amount to shift the threshold chosen by the program by before segmentation.
- Lower values lower the threshold and make the segmentation more sensitive, if it is too low for an image grains may be split into multiple
- Higher values raise the threshold making the segmentation less sensitive, if it's too high then multiple grains may be joined together
- When testing for an ideal offset, change the value by ~0.5-1 (or less for fine tuning)

##### `image_set`

The image set list can be used to turn on or off different `.png` outputs. Comment or uncomment image names in this list to add or remove them from the output.

##### `number_grains`

Add grain numbers to the output images with mask overlays.

##### `height_scale`

Option to include a colourbar height scale on the side of relevant images.

#### Adjust the values in the code below and then re-run the notebook to see how that changes things:

In [ ]:
# The file extension of the input image(s).
file_ext = ".spm"

# If you are inputting images without a background material frequency splitting can be disabled here.
run_splitting = True # Options: True, False

# Adjust these bounds to increase the difference if no cutoff can be found
cutoff_bounds = [-1, 1]

# Adjust this value if the background material is still visible after frequency splitting or the image is too flat.
# DECREATE if the image still contains the background material
# INCREASE if the image is too flat
min_rms = 10

# threshold_offset only applies to traditional segmentation
# DECREASE if grains are being combined into one
# INCREASE if grains are being split into multiple
threshold_offset = -0.5

# Comment or uncomment image names to select which ones to output
image_set = [
    "highpass_mask",
    "highpass",
    "lowpass",
    "mask",
    "numbered_grains",
    "original_mask",
    "original",
    "rgb_grains",
    "smears",
]

# Add grain numbers to the output images with mask overlays
number_grains = True

# Include a colourbar height scale
height_scale = True

---
## PerovStats Process

You do not need to change any of the below code, just scroll through as the program runs to see the generated images/ log messages through the process.

#### First, the files are loaded and the original heightmap image is extracted.

In [ ]:
setup_logger()

default_config_path = Path("../src/perovstats/default_config.yaml")
with default_config_path.open() as f:
    config = yaml.safe_load(f)

if config_path.resolve() != default_config_path.resolve():
    if config_path.exists():
        with config_path.open() as f:
            custom_config = yaml.safe_load(f)

        config = deep_merge(config, custom_config)
    else:
        logger.error("Error: custom configuration file could not be found. Please check the set filepath above.")

time_start = time.perf_counter()

config["file_ext"] = file_ext

# Load scans
load_config = config["loading"]
loadscans = LoadScans(img_file, config)
try:
    loadscans.get_data()
except ValueError as e:
    logger.warning(e)
    logger.warning(f"Channel {load_config['channel']} not found in file. Please ensure the config option is correct and all files contain the required channel.")
image_dicts = loadscans.img_dict

config["output"]["height_scale"] = height_scale
config["output"]["image_set"] = image_set

config["fourier"]["run"] = run_splitting
config["fourier"]["min_rms"] = min_rms
config["fourier"]["cutoff_bounds"] = cutoff_bounds

segmentation_method = "traditional"
config["segmentation"]["segmentation_method"] = segmentation_method
config["segmentation"]["traditional"]["threshold_offset"] = threshold_offset

config["output"]["number_grains"] = number_grains

cmap = config["colour_scheme"]

# Create the dataclasses for the whole process and for each image found
perovstats_object = PerovStats(config=config, images=[])
for filename, topostats_object in image_dicts.items():
    image_data = ImageData(
        success=True,
        filename=filename,
        pixel_to_nm_scaling=topostats_object.pixel_to_nm_scaling,
        image_original=topostats_object.image_original,
        image_flattened=None)
    perovstats_object.images.append(image_data)

logger.info("----------------------------------------------------------")
logger.info(f"Loaded {len(perovstats_object.images)} images")

idx = 0
image_object = perovstats_object.images[0]

plt.imshow(image_object.image_original, cmap=cmap)
plt.title("Original image")
plt.axis("off")
plt.show()

In [ ]:
logger.info("----------------------------------------------------------")
logger.info(f"Processing {image_object.filename} ({idx+1}/{len(perovstats_object.images)})")
logger.info("----------------------------------------------------------")
logger.debug(f"[{image_object.filename}] : Image dimensions: {image_object.image_original.shape}")
logger.debug(f"[{image_object.filename}] : pixel_to_nm_scaling: {image_object.pixel_to_nm_scaling}")

---
#### A fourier transform is then performed on the image which splits it into two sections, one containing the perovskite topology and one containing the silicon topology (to be discarded):

The frequency at which the image is split can be adjusted by the slider that will appear once the code cell is run. A preview of the resultant high and low pass will also be shown below. Once a suitable setting has been found move on to the next cells.

In [ ]:
def preview_split(split_freq):
    split_frequencies(perovstats_object.config, image_object, split_freq=split_freq)

    # Show the split images
    _, ax = plt.subplots(1, 2)

    ax[0].imshow(image_object.high_pass, cmap=cmap)
    ax[0].set_title("High passed image (Perovskite)")
    ax[0].axis("off")

    ax[1].imshow(image_object.low_pass, cmap=cmap)
    ax[1].set_title("Low passed image (Silicon)")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()


split_frequencies(perovstats_object.config, image_object)

interact(
    preview_split,
    split_freq=FloatSlider(
        min=0.001,
        max=0.5,
        step=0.001,
        value=image_object.cutoff,
        description='Cutoff Frequency:',
        layout=Layout(width='60%'),
        readout_format='.3f',
        continuous_update=False,
    ),
)

---
#### The high-passed image must now be segmented in order to find individual grain outlines. If `cellpose` has been selected as the segmentation method the program will use a machine learning model and may take a few minutes depending on your device.

##### For the quickest results, use a computer that has an Nvidia GPU

When cellpose mask segmentation completes for an image warnings may appear. As long as the green `SUCCESS` log appears under it, you can ignore these warnings.

For this demo both methods are used and results will be displayed side by side.

In [ ]:
from IPython.display import Markdown

cmap = config["colour_scheme"]
get_cmap = cm.get_cmap(cmap)

perovstats_object_trad = copy.deepcopy(perovstats_object)
perovstats_object_trad.config["segmentation"]["segmentation_method"] = "traditional"
image_object_trad = copy.deepcopy(perovstats_object_trad.images[0])
perovstats_object_cellpose = copy.deepcopy(perovstats_object)
perovstats_object_cellpose.config["segmentation"]["segmentation_method"] = "cellpose"
image_object_cellpose = copy.deepcopy(perovstats_object_cellpose.images[0])

# Generate grain mask of the high-passed image
trad_start_time = time.perf_counter()
display(Markdown("### Traditional segmentation"))
segment_image(perovstats_object_trad.config, image_object_trad)
trad_end_time = time.perf_counter()
trad_time = format_time(trad_end_time - trad_start_time)

cell_start_time = time.perf_counter()
display(Markdown("### Cellpose ML segmentation"))
segment_image(perovstats_object_cellpose.config, image_object_cellpose)
cell_end_time = time.perf_counter()
cell_time = format_time(cell_end_time - cell_start_time)

# Display the generated masks
fig, ax = plt.subplots(1, 2)

ax[0].imshow(image_object_trad.mask, cmap=cmap)
ax[0].set_title(f"Traditionally generated mask ({trad_time})")
ax[0].axis("off")

ax[1].imshow(image_object_cellpose.mask, cmap=cmap)
ax[1].set_title(f"Cellpose generated mask ({cell_time})")
ax[1].axis("off")

plt.tight_layout()
plt.show()

# Display the generated masks overlayed onto the high pass image
fig, ax = plt.subplots(1, 2)

mask_overlay_trad = image_object_trad.high_pass
norm_highpass = normalise_array(mask_overlay_trad)
rgba_highpass = get_cmap(norm_highpass)
mask_overlay_trad = rgba_highpass[..., :3]
mask_overlay_trad[image_object_trad.mask > 0] = [0, 0, 1]

mask_overlay_cellpose = image_object_cellpose.high_pass
norm_highpass = normalise_array(mask_overlay_cellpose)
rgba_highpass = get_cmap(norm_highpass)
mask_overlay_cellpose = rgba_highpass[..., :3]
mask_overlay_cellpose[image_object_cellpose.mask > 0] = [0, 0, 1]

ax[0].imshow(mask_overlay_trad, cmap=cmap)
ax[0].set_title("Mask overlay traditional")
ax[0].axis("off")

ax[1].imshow(mask_overlay_cellpose, cmap=cmap)
ax[1].set_title("Mask overlay cellpose")
ax[1].axis("off")

plt.tight_layout()
plt.show()

---
#### Smear areas are errors in the AFM scan from the tip not being able to dip low enough after a peak, resulting in horizontal lines which should not be counted in the data. They are identified and grains within these areas are removed:

In [ ]:
# Find smear areas to be ignored/ removed
find_smear_areas(perovstats_object_trad.config, image_object_trad)
find_smear_areas(perovstats_object_cellpose.config, image_object_cellpose)

---
##### The mask (which is currently a collection of thin lines) should then be analysed in order to identify individual grains.
Once this has been done grains touching either the edge of the image or an identified smear area can be removed.

In [ ]:
# Identify individual grains from mask and generate statistics on them
find_grains(perovstats_object_trad.config, image_object_trad) 
find_grains(perovstats_object_cellpose.config, image_object_cellpose)

Stats can also now be taken from these grains and displayed in graphs:

In [ ]:
highpass_overlay_trad = image_object_trad.high_pass
norm_highpass = normalise_array(highpass_overlay_trad)
rgba_highpass = get_cmap(norm_highpass)
highpass_overlay_trad = rgba_highpass[..., :3]
highpass_overlay_trad[image_object_trad.mask > 0] = [0, 0, 1]

highpass_overlay_cellpose = image_object_cellpose.high_pass
norm_highpass = normalise_array(highpass_overlay_cellpose)
rgba_highpass = get_cmap(norm_highpass)
highpass_overlay_cellpose = rgba_highpass[..., :3]
highpass_overlay_cellpose[image_object_cellpose.mask > 0] = [0, 0, 1]

original_overlay_trad = image_object_trad.image_original
norm_original = normalise_array(original_overlay_trad)
rgba_original = get_cmap(norm_original)
original_overlay_trad = rgba_original[..., :3]
original_overlay_trad[image_object_trad.mask > 0] = [0, 0, 1]

original_overlay_cellpose = image_object_cellpose.image_original
norm_original = normalise_array(original_overlay_cellpose)
rgba_original = get_cmap(norm_original)
original_overlay_cellpose = rgba_original[..., :3]
original_overlay_cellpose[image_object_cellpose.mask > 0] = [0, 0, 1]

# Display data and statistics on the processed image and grains
display(Markdown("## Traditional segmentation"))
fig, ax = plt.subplots(3, 2, figsize=(10, 12))

ax[0,0].imshow(highpass_overlay_trad, cmap=cmap)
ax[0,0].set_title("Highpassed image overlayed with final mask")
ax[0,0].axis("off")

ax[0,1].imshow(image_object_trad.mask_rgb, cmap=cmap)
ax[0,1].set_title("Coloured grains with smears removed")
ax[0,1].axis("off")

ax[1,0].imshow(image_object_trad.image_original, cmap=cmap)
ax[1,0].set_title("Original Image")
ax[1,0].axis("off")

ax[1,1].imshow(original_overlay_trad, cmap=cmap)
ax[1,1].set_title("Original image overlayed with final mask")
ax[1,1].axis("off")

sns.histplot(image_object_trad.mask_areas, bins='auto', kde=True, log_scale=True, color='skyblue', edgecolor='black', ax=ax[2,0])
ax[2,0].set_xlabel('Values')
ax[2,0].set_ylabel('Frequency')
ax[2,0].set_title('Grain areas nm²')

sns.histplot(image_object_trad.circularity_data, bins='auto', kde=True, color='skyblue', edgecolor='black', ax=ax[2,1])
ax[2,1].set_xlabel('Values')
ax[2,1].set_ylabel('Frequency')
ax[2,1].set_title('Grain circularities (0-1)')

plt.tight_layout()
plt.show()


display(Markdown("## Cellpose (ML) segmentation"))
fig, ax = plt.subplots(3, 2, figsize=(10, 12))

ax[0,0].imshow(highpass_overlay_cellpose, cmap=cmap)
ax[0,0].set_title("Highpassed image overlayed with final mask")
ax[0,0].axis("off")

ax[0,1].imshow(image_object_cellpose.mask_rgb, cmap=cmap)
ax[0,1].set_title("Coloured grains with smears removed")
ax[0,1].axis("off")

ax[1,0].imshow(image_object_cellpose.image_original, cmap=cmap)
ax[1,0].set_title("Original Image")
ax[1,0].axis("off")

ax[1,1].imshow(original_overlay_cellpose, cmap=cmap)
ax[1,1].set_title("Original image overlayed with final mask")
ax[1,1].axis("off")

sns.histplot(image_object_cellpose.mask_areas, bins='auto', kde=True, log_scale=True, color='skyblue', edgecolor='black', ax=ax[2,0])
ax[2,0].set_xlabel('Values')
ax[2,0].set_ylabel('Frequency')
ax[2,0].set_title('Grain areas nm²')

sns.histplot(image_object_cellpose.circularity_data, bins='auto', kde=True, color='skyblue', edgecolor='black', ax=ax[2,1])
ax[2,1].set_xlabel('Values')
ax[2,1].set_ylabel('Frequency')
ax[2,1].set_title('Grain circularities (0-1)')

plt.tight_layout()
plt.show()

---
##### Data (including the config options used in the run) are now exported to the output directory chosen at the top of this notebook:

In [ ]:
logger.info(f"[{image_object_trad.filename}] : *** Exporting data ***")
# Save image and grain data to their own .csv file
image_df = pd.DataFrame([image_object_trad.to_dict()])
grains_list = []
for grain in image_object_trad.grains.values():
    grains_list.append(grain.to_dict())
grain_df = pd.DataFrame(grains_list)

save_images(perovstats_object.config, image_object_cellpose, variation="cellpose")
save_images(perovstats_object.config, image_object_trad, variation="traditional")

output_filename = f"{output_dir}/{image_object_trad.filename}/image_statistics.csv"
save_to_csv(image_df, output_filename)
output_filename = f"{output_dir}/{image_object_trad.filename}/grain_statistics.csv"
save_to_csv(grain_df, output_filename)
# Save the config settings in a .yaml
output_filename = Path(output_dir) / Path(image_object_trad.filename) / "config.yaml"
save_config(perovstats_object_trad.config, output_filename)

logger.info(
    f"[{image_object.filename}] : Exported data and config to {Path(output_dir) / Path(image_object_trad.filename)}",
)

time_end = time.perf_counter()
time_taken = format_time(time_end - time_start)
time_per_image = format_time((time_end - time_start) / len(perovstats_object.images))
completion_message(perovstats_object, time_taken, time_per_image)